In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix


In [2]:
# Load dataset
data = pd.read_csv('data_finish.csv', delimiter = ',')

# Drop unnecessary columns
data.drop(['ddd_car', 'station_id', 'station_name', 'region_name', 'date'], axis=1, inplace=True)

# Display the first few rows of the dataset
print(data.head())

     Tn    Tx  Tavg  RH_avg    RR   ss  ff_x  ddd_x  ff_avg  flood
0  26.0  34.8  28.6    81.0   NaN  5.8   5.0  280.0     2.0      0
1  25.6  33.2  27.0    88.0   1.6  8.7   4.0  290.0     2.0      1
2  24.4  34.9  28.1    80.0  33.8  5.4   4.0  280.0     2.0      1
3  24.8  33.6  29.2    81.0   NaN  6.6   3.0  200.0     1.0      0
4  25.8  33.6  26.7    91.0   NaN  3.2   3.0  180.0     1.0      0


In [3]:
# Menggantikan nilai NaN dengan rata-rata kolom
data.fillna(data.mean(), inplace=True)

print(data.isnull().sum())

Tn        0
Tx        0
Tavg      0
RH_avg    0
RR        0
ss        0
ff_x      0
ddd_x     0
ff_avg    0
flood     0
dtype: int64


In [4]:
X = data[['Tn', 'Tx', 'Tavg', 'RH_avg', 'RR', 'ss', 'ff_x']]
y = data['flood']

In [5]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [6]:
# Membagi dataset menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [7]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

c:\Python\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
# Menyusun model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [9]:
# Melatih model
history = model.fit(X_train, y_train, epochs=50, batch_size=64, validation_split=0.2)

Epoch 1/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8202 - loss: 0.5024 - val_accuracy: 0.9257 - val_loss: 0.2337
Epoch 2/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9151 - loss: 0.2806 - val_accuracy: 0.9257 - val_loss: 0.2086
Epoch 3/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9289 - loss: 0.2445 - val_accuracy: 0.9277 - val_loss: 0.2062
Epoch 4/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9275 - loss: 0.2276 - val_accuracy: 0.9297 - val_loss: 0.2061
Epoch 5/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9216 - loss: 0.2329 - val_accuracy: 0.9287 - val_loss: 0.2063
Epoch 6/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9335 - loss: 0.2064 - val_accuracy: 0.9287 - val_loss: 0.2075
Epoch 7/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9250 - loss: 0.2207 - val_accuracy: 0.9277 - val_loss: 0.2062
Epoch 8/50
64/64 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9330 - loss: 0.2054 - val_accuracy: 0.9277 - val_loss:

In [10]:
# Evaluasi model pada data uji
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_accuracy}')

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9183 - loss: 0.2410
Test Accuracy: 0.9136291742324829


In [11]:
# Membuat prediksi pada data uji
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs > 0.5).astype(int)

40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [12]:
# Menampilkan confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Menampilkan laporan klasifikasi
print("Classification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[1151    3]
 [ 106    2]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.95      1154
           1       0.40      0.02      0.04       108

    accuracy                           0.91      1262
   macro avg       0.66      0.51      0.50      1262
weighted avg       0.87      0.91      0.88      1262



In [13]:
model.save('mymodel.h5')